In [1]:
import pandas as pd
from pathlib import Path
from collections import Counter

In [2]:
DATA_DIR = Path("../training_datasets")

S1_PATH = DATA_DIR / "train_source1.tsv"
S2_PATH = DATA_DIR / "train_source2.tsv"
S3_PATH = DATA_DIR / "train_source3.tsv"
GT_PATH = DATA_DIR / "train_ground_truth.tsv"

CHUNK_SIZE = 100_000

In [41]:
def profile_source(path, chunk_size=100_000):
    total_rows = 0

    missing_counts = Counter()
    country_counts = Counter()

    unique_ids = set()
    duplicate_ids = 0

    name_lengths = []
    address_lengths = []

    unique_names = set()
    unique_addresses = set()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        chunksize=chunk_size
    ):
        total_rows += len(chunk)

        # Missing values
        missing_counts.update(
            chunk.isna().sum().to_dict()
        )

        # Countries
        country_counts.update(
            chunk["country"].fillna("<MISSING>").value_counts().to_dict()
        )

        # Entity IDs
        ids = chunk["entity_id"].dropna()

        duplicate_ids += ids.duplicated().sum()

        # Check duplicates across chunks too
        for entity_id in ids:
            if entity_id in unique_ids:
                duplicate_ids += 1
            else:
                unique_ids.add(entity_id)

        # Name statistics
        names = chunk["business_name"].fillna("")

        name_lengths.extend(names.str.len().tolist())
        unique_names.update(names[names != ""])

        # Address statistics
        addresses = chunk["business_address"].fillna("")

        address_lengths.extend(addresses.str.len().tolist())
        unique_addresses.update(addresses[addresses != ""])

    print(f"Rows: {total_rows:,}")
    print(f"Columns: 4")

    print("\nMissing values:")
    for column, count in missing_counts.items():
        print(f"  {column}: {count:,}")

    print("\nEntity IDs:")
    print(f"  Unique IDs: {len(unique_ids):,}")
    print(f"  Duplicate IDs: {duplicate_ids:,}")

    print("\nCountries:")
    for country, count in country_counts.most_common():
        print(f"  {country}: {count:,}")

    print("\nBusiness name:")
    print(f"  Unique names: {len(unique_names):,}")
    print(f"  Average length: {sum(name_lengths) / len(name_lengths):.2f}")
    print(f"  Min length: {min(name_lengths)}")
    print(f"  Max length: {max(name_lengths)}")

    print("\nBusiness address:")
    print(f"  Unique addresses: {len(unique_addresses):,}")
    print(f"  Average length: {sum(address_lengths) / len(address_lengths):.2f}")
    print(f"  Min length: {min(address_lengths)}")
    print(f"  Max length: {max(address_lengths)}")

    return {
        "rows": total_rows,
        "unique_ids": len(unique_ids),
        "duplicate_ids": duplicate_ids,
        "country_counts": country_counts,
        "missing_counts": missing_counts,
    }

In [42]:
s1_profile = profile_source(S1_PATH)

Rows: 2,206,821
Columns: 4

Missing values:
  entity_id: 0
  business_name: 0
  business_address: 0
  country: 0

Entity IDs:
  Unique IDs: 2,206,821
  Duplicate IDs: 0

Countries:
  US: 1,323,633
  India: 883,188

Business name:
  Unique names: 1,539,229
  Average length: 24.03
  Min length: 3
  Max length: 105

Business address:
  Unique addresses: 2,130,606
  Average length: 52.07
  Min length: 11
  Max length: 256


In [43]:
s2_profile = profile_source(S2_PATH)

Rows: 5,034,616
Columns: 4

Missing values:
  entity_id: 0
  business_name: 2
  business_address: 168,967
  country: 0

Entity IDs:
  Unique IDs: 5,034,616
  Duplicate IDs: 0

Countries:
  US: 3,016,817
  India: 2,017,799

Business name:
  Unique names: 4,402,008
  Average length: 25.10
  Min length: 0
  Max length: 104

Business address:
  Unique addresses: 4,337,261
  Average length: 46.23
  Min length: 0
  Max length: 249


In [44]:
s3_profile = profile_source(S3_PATH)

Rows: 5,285,603
Columns: 4

Missing values:
  entity_id: 0
  business_name: 13
  business_address: 175,916
  country: 0

Entity IDs:
  Unique IDs: 5,285,603
  Duplicate IDs: 0

Countries:
  US: 3,170,056
  India: 2,115,547

Business name:
  Unique names: 4,651,608
  Average length: 25.20
  Min length: 0
  Max length: 123

Business address:
  Unique addresses: 4,632,764
  Average length: 46.71
  Min length: 0
  Max length: 240


# Profiling Ground Truth Function

In [45]:
def get_entity_ids(path, chunk_size=100_000):
    ids = set()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        usecols=["entity_id"],
        chunksize=chunk_size
    ):
        ids.update(chunk["entity_id"].dropna())

    return ids

In [46]:
s2_ids = get_entity_ids(S2_PATH)
s3_ids = get_entity_ids(S3_PATH)

print(f"S2 IDs: {len(s2_ids):,}")
print(f"S3 IDs: {len(s3_ids):,}")

S2 IDs: 5,034,616
S3 IDs: 5,285,603


In [47]:
def profile_ground_truth(path, s2_ids, s3_ids):
    total_rows = 0

    singleton_count = 0
    match_count_distribution = Counter()

    s2_match_count = 0
    s3_match_count = 0
    unknown_match_count = 0

    source1_ids = set()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        chunksize=100_000
    ):
        total_rows += len(chunk)

        # Source 1 IDs
        source1_ids.update(
            chunk["source1_entity_id"].dropna()
        )

        for matched_ids in chunk["matched_entity_ids"].fillna(""):
            matched_ids = matched_ids.strip()

            # Singleton
            if not matched_ids:
                singleton_count += 1
                match_count_distribution[0] += 1
                continue

            ids = [
                entity_id.strip()
                for entity_id in matched_ids.split(",")
                if entity_id.strip()
            ]

            match_count_distribution[len(ids)] += 1

            for entity_id in ids:
                if entity_id in s2_ids:
                    s2_match_count += 1

                elif entity_id in s3_ids:
                    s3_match_count += 1

                else:
                    unknown_match_count += 1

    print(f"Rows: {total_rows:,}")

    print("\nUnique Source 1 IDs:")
    print(f"  {len(source1_ids):,}")

    print("\nMatch count distribution:")

    for count, frequency in sorted(match_count_distribution.items()):
        print(f"  {count} matches: {frequency:,}")

    print("\nSingletons:")
    print(f"  {singleton_count:,}")
    print(f"  Singleton rate: {singleton_count / total_rows:.2%}")

    print("\nMatches by source:")
    print(f"  S2: {s2_match_count:,}")
    print(f"  S3: {s3_match_count:,}")
    print(f"  Unknown IDs: {unknown_match_count:,}")

    return {
        "rows": total_rows,
        "unique_source1_ids": len(source1_ids),
        "singleton_count": singleton_count,
        "singleton_rate": singleton_count / total_rows,
        "match_distribution": match_count_distribution,
        "s2_matches": s2_match_count,
        "s3_matches": s3_match_count,
        "unknown_matches": unknown_match_count,
    }

In [48]:
gt_profile = profile_ground_truth(
    GT_PATH,
    s2_ids,
    s3_ids
)

Rows: 2,206,821

Unique Source 1 IDs:
  2,206,821

Match count distribution:
  0 matches: 123,247
  1 matches: 119,157
  2 matches: 375,212
  3 matches: 530,841
  4 matches: 484,115
  5 matches: 321,957
  6 matches: 164,868
  7 matches: 63,968
  8 matches: 18,680
  9 matches: 4,205
  10 matches: 534
  11 matches: 37

Singletons:
  123,247
  Singleton rate: 5.58%

Matches by source:
  S2: 3,693,619
  S3: 3,944,746
  Unknown IDs: 0


# Classifying Ground Truth Matches by Source

In [49]:
from collections import Counter

def profile_match_sources(path, s2_ids, s3_ids, chunk_size=100_000):
    categories = Counter()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        chunksize=chunk_size
    ):
        for matched_ids in chunk["matched_entity_ids"].fillna(""):

            matched_ids = matched_ids.strip()

            # No matches
            if not matched_ids:
                categories["neither"] += 1
                continue

            ids = [
                entity_id.strip()
                for entity_id in matched_ids.split(",")
                if entity_id.strip()
            ]

            has_s2 = False
            has_s3 = False

            for entity_id in ids:
                if entity_id in s2_ids:
                    has_s2 = True
                elif entity_id in s3_ids:
                    has_s3 = True

            if has_s2 and has_s3:
                categories["both"] += 1
            elif has_s2:
                categories["s2_only"] += 1
            elif has_s3:
                categories["s3_only"] += 1
            else:
                categories["unknown"] += 1

    total = sum(categories.values())

    print(f"Total S1 businesses: {total:,}")

    print("\nMatch source categories:")

    for category in ["s2_only", "s3_only", "both", "neither", "unknown"]:
        count = categories[category]
        percentage = count / total * 100

        print(
            f"  {category:10} → "
            f"{count:>10,} ({percentage:.2f}%)"
        )

    return categories

In [50]:
source_categories = profile_match_sources(
    GT_PATH,
    s2_ids,
    s3_ids
)

Total S1 businesses: 2,206,821

Match source categories:
  s2_only    →    143,029 (6.48%)
  s3_only    →    164,498 (7.45%)
  both       →  1,776,047 (80.48%)
  neither    →    123,247 (5.58%)
  unknown    →          0 (0.00%)


# Disjointness of Source 2 and Source 3

In [52]:
def get_ids(path):
    ids = set()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id"],
        chunksize=CHUNK_SIZE,
        dtype={"entity_id": "string"}
    ):
        ids.update(chunk["entity_id"].dropna())

    return ids


print("Reading S2 IDs...")
s2_ids = get_ids(S2_PATH)

print("Reading S3 IDs...")
s3_ids = get_ids(S3_PATH)

print("\n--- DISJOINTNESS CHECK ---")

print(f"S2 unique IDs: {len(s2_ids):,}")
print(f"S3 unique IDs: {len(s3_ids):,}")

overlap = s2_ids & s3_ids

print(f"S2 ∩ S3: {len(overlap):,}")

print(f"S2-only: {len(s2_ids - s3_ids):,}")
print(f"S3-only: {len(s3_ids - s2_ids):,}")

print(
    f"\nOverlap rate relative to S2: "
    f"{len(overlap) / len(s2_ids) * 100:.2f}%"
)

print(
    f"Overlap rate relative to S3: "
    f"{len(overlap) / len(s3_ids) * 100:.2f}%"
)

Reading S2 IDs...
Reading S3 IDs...

--- DISJOINTNESS CHECK ---
S2 unique IDs: 5,034,616
S3 unique IDs: 5,285,603
S2 ∩ S3: 0
S2-only: 5,034,616
S3-only: 5,285,603

Overlap rate relative to S2: 0.00%
Overlap rate relative to S3: 0.00%


In [53]:
# ============================================================
# COUNTRY CONSISTENCY CHECK
# S1 ↔ S2 and S1 ↔ S3 true matches from ground truth
# ============================================================

import pandas as pd

print("Loading ground truth...")

# Only load the columns we need
gt = pd.read_csv(
    GT_PATH,
    sep="\t",
    usecols=["source1_entity_id", "matched_entity_ids"],
    dtype=str
)

print(f"Ground truth rows: {len(gt):,}")


# ------------------------------------------------------------
# 1. EXPLODE GROUND TRUTH INTO INDIVIDUAL MATCH PAIRS
# ------------------------------------------------------------

pairs = []

for _, row in gt.iterrows():

    s1_id = row["source1_entity_id"]
    matched = row["matched_entity_ids"]

    if pd.isna(s1_id) or pd.isna(matched):
        continue

    for matched_id in str(matched).split(","):

        matched_id = matched_id.strip()

        if matched_id.startswith("S2-"):
            pairs.append((s1_id, matched_id, "S2"))

        elif matched_id.startswith("S3-"):
            pairs.append((s1_id, matched_id, "S3"))


pairs = pd.DataFrame(
    pairs,
    columns=["s1_id", "matched_id", "source"]
)

print(f"Total true match pairs: {len(pairs):,}")
print(f"S1 ↔ S2 pairs: {(pairs['source'] == 'S2').sum():,}")
print(f"S1 ↔ S3 pairs: {(pairs['source'] == 'S3').sum():,}")


# ------------------------------------------------------------
# 2. GET ONLY THE IDS WE NEED
# ------------------------------------------------------------

s1_ids = set(pairs["s1_id"])
s2_ids = set(pairs.loc[pairs["source"] == "S2", "matched_id"])
s3_ids = set(pairs.loc[pairs["source"] == "S3", "matched_id"])


# ------------------------------------------------------------
# 3. LOOK UP COUNTRIES FROM EACH SOURCE
#    Chunked so we don't unnecessarily load 5M+ rows
# ------------------------------------------------------------

def load_country_lookup(path, wanted_ids, label):

    lookup = {}

    print(f"\nReading {label} countries...")

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id", "country"],
        dtype=str,
        chunksize=CHUNK_SIZE
    ):

        mask = chunk["entity_id"].isin(wanted_ids)

        if mask.any():

            selected = chunk.loc[mask, ["entity_id", "country"]]

            for entity_id, country in selected.itertuples(index=False):
                lookup[entity_id] = country

    print(f"{label} IDs found: {len(lookup):,} / {len(wanted_ids):,}")

    return lookup


s1_country = load_country_lookup(
    S1_PATH,
    s1_ids,
    "S1"
)

s2_country = load_country_lookup(
    S2_PATH,
    s2_ids,
    "S2"
)

s3_country = load_country_lookup(
    S3_PATH,
    s3_ids,
    "S3"
)


# ------------------------------------------------------------
# 4. ATTACH COUNTRIES TO MATCH PAIRS
# ------------------------------------------------------------

pairs["s1_country"] = pairs["s1_id"].map(s1_country)

pairs["matched_country"] = pairs.apply(
    lambda row:
        s2_country.get(row["matched_id"])
        if row["source"] == "S2"
        else s3_country.get(row["matched_id"]),
    axis=1
)


# ------------------------------------------------------------
# 5. NORMALIZE COUNTRY VALUES FOR COMPARISON
# ------------------------------------------------------------

pairs["s1_country_norm"] = (
    pairs["s1_country"]
    .fillna("")
    .str.strip()
    .str.lower()
)

pairs["matched_country_norm"] = (
    pairs["matched_country"]
    .fillna("")
    .str.strip()
    .str.lower()
)


# ------------------------------------------------------------
# 6. CHECK CONSISTENCY
# ------------------------------------------------------------

# Only compare rows where BOTH countries are available
pairs["countries_available"] = (
    (pairs["s1_country_norm"] != "") &
    (pairs["matched_country_norm"] != "")
)

pairs["country_match"] = (
    pairs["countries_available"] &
    (
        pairs["s1_country_norm"]
        == pairs["matched_country_norm"]
    )
)


# ------------------------------------------------------------
# 7. PRINT RESULTS
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("COUNTRY CONSISTENCY")
print("=" * 60)


for source in ["S2", "S3"]:

    subset = pairs[pairs["source"] == source]

    available = subset["countries_available"].sum()

    mismatches = (
        subset.loc[subset["countries_available"], "country_match"]
        == False
    ).sum()

    matches = (
        subset.loc[subset["countries_available"], "country_match"]
        == True
    ).sum()

    missing = len(subset) - available

    mismatch_rate = (
        mismatches / available * 100
        if available > 0
        else 0
    )

    print(f"\nS1 ↔ {source}")
    print("-" * 30)
    print(f"True match pairs:       {len(subset):,}")
    print(f"Countries available:    {available:,}")
    print(f"Country matches:        {matches:,}")
    print(f"Country mismatches:     {mismatches:,}")
    print(f"Missing country values: {missing:,}")
    print(f"Mismatch rate:          {mismatch_rate:.4f}%")


# ------------------------------------------------------------
# 8. SHOW ACTUAL MISMATCHES
# ------------------------------------------------------------

mismatches = pairs[
    pairs["countries_available"] &
    (~pairs["country_match"])
]

print("\n")
print("=" * 60)
print("COUNTRY MISMATCH DETAILS")
print("=" * 60)

if len(mismatches) == 0:

    print("No country mismatches found.")

else:

    print(mismatches[
        [
            "s1_id",
            "s1_country",
            "matched_id",
            "matched_country",
            "source"
        ]
    ].to_string(index=False))

Loading ground truth...
Ground truth rows: 2,206,821
Total true match pairs: 7,638,365
S1 ↔ S2 pairs: 3,693,619
S1 ↔ S3 pairs: 3,944,746

Reading S1 countries...
S1 IDs found: 2,083,574 / 2,083,574

Reading S2 countries...
S2 IDs found: 3,693,619 / 3,693,619

Reading S3 countries...
S3 IDs found: 3,944,746 / 3,944,746


COUNTRY CONSISTENCY

S1 ↔ S2
------------------------------
True match pairs:       3,693,619
Countries available:    3,693,619
Country matches:        3,693,619
Country mismatches:     0
Missing country values: 0
Mismatch rate:          0.0000%

S1 ↔ S3
------------------------------
True match pairs:       3,944,746
Countries available:    3,944,746
Country matches:        3,944,746
Country mismatches:     0
Missing country values: 0
Mismatch rate:          0.0000%


COUNTRY MISMATCH DETAILS
No country mismatches found.


# Name Noice Check

In [5]:
# GT schema check + samples
gt = pd.read_csv(GT_PATH, sep='\t', dtype=str)
print("columns:", gt.columns.tolist())
print("\n=== GT head ===")
print(gt.head(10).to_string())
print("\nnull matched_entity_ids:", gt['matched_entity_ids'].isna().sum())
print("empty-string matched_entity_ids:", (gt['matched_entity_ids'].fillna('') == '').sum())

# explode a few rows to see the actual matched ID strings
sample = gt.head(5).copy()
sample['matched_list'] = sample['matched_entity_ids'].fillna('').str.split(',')
print("\n=== exploded sample ===")
for _, row in sample.iterrows():
    print(row['source1_entity_id'], "→", row['matched_list'])

# S2 / S3 entity_id format (first 10)
print("\n=== S2 first 10 ===")
s2 = pd.read_csv(S2_PATH, sep='\t', usecols=['entity_id', 'business_name'], dtype=str, nrows=10)
print(s2.to_string())
print("S2 entity_id samples:", s2['entity_id'].tolist())

print("\n=== S3 first 10 ===")
s3 = pd.read_csv(S3_PATH, sep='\t', usecols=['entity_id', 'business_name'], dtype=str, nrows=10)
print(s3.to_string())
print("S3 entity_id samples:", s3['entity_id'].tolist())

columns: ['source1_entity_id', 'matched_entity_ids']

=== GT head ===
  source1_entity_id                                                             matched_entity_ids
0         S1-965667                S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364
1       S1-55344266                            S2-249013014,S2-197070651,S3-478195123,S3-384364074
2      S1-343815751                                         S2-790675320,S2-479876582,S3-878454467
3      S1-656753428                                          S2-153058913,S2-24659151,S3-679606215
4      S1-102811957  S2-478959098,S2-553508714,S2-625774905,S3-728090388,S3-928796641,S3-449308785
5       S1-18727616                            S2-755677256,S3-187831601,S3-641489370,S3-476250621
6      S1-318373630                                                      S2-660036492,S3-804600254
7       S1-86989137                                                      S3-274817120,S3-312496301
8       S1-29845983                    

In [6]:

# ---------- 1. Build name lookups (chunked, dtype=str) ----------
def build_name_lookup(path, id_col='entity_id', name_col='business_name'):
    lookup = {}
    for chunk in pd.read_csv(path, sep='\t', usecols=[id_col, name_col],
                             dtype=str, chunksize=CHUNK_SIZE):
        # keep first occurrence if any dups (there shouldn't be)
        chunk = chunk.dropna(subset=[id_col])
        for eid, name in zip(chunk[id_col], chunk[name_col]):
            if eid not in lookup:
                lookup[eid] = name if pd.notna(name) else ''
    return lookup

print("Loading S1 names...")
s1_names = build_name_lookup(S1_PATH)
print(f"  S1 names: {len(s1_names):,}")

print("Loading S2 names...")
s2_names = build_name_lookup(S2_PATH)
print(f"  S2 names: {len(s2_names):,}")

print("Loading S3 names...")
s3_names = build_name_lookup(S3_PATH)
print(f"  S3 names: {len(s3_names):,}")

# ---------- 2. Explode GT into pairs ----------
gt = pd.read_csv(GT_PATH, sep='\t', dtype=str)
gt['matched_list'] = gt['matched_entity_ids'].fillna('').str.split(',')
pairs = gt.explode('matched_list')
pairs = pairs[pairs['matched_list'].str.len() > 0].copy()
pairs = pairs.rename(columns={'matched_list': 'matched_id'})
print(f"\nTrue match pairs after explode: {len(pairs):,}")

# ---------- 3. Attach names ----------
def get_name(mid):
    if mid.startswith('S2-'):
        return s2_names.get(mid, None)
    if mid.startswith('S3-'):
        return s3_names.get(mid, None)
    return None

pairs['s1_name'] = pairs['source1_entity_id'].map(s1_names)
pairs['matched_name'] = pairs['matched_id'].map(get_name)

both = pairs.dropna(subset=['s1_name', 'matched_name']).copy()
print(f"Pairs with both names present: {len(both):,}")
print(f"  missing S1 name: {pairs['s1_name'].isna().sum():,}")
print(f"  missing matched name: {pairs['matched_name'].isna().sum():,}")

# ---------- 4. Normalization helpers ----------
SUFFIXES = [
    r'\binc\b', r'\bincorporated\b', r'\bltd\b', r'\blimited\b',
    r'\bllc\b', r'\bcorp\b', r'\bcorporation\b', r'\bco\b', r'\bcompany\b',
    r'\bpvt\b', r'\bprivate\b', r'\bllp\b', r'\bplc\b'
]
SUFFIX_RE = re.compile(r'(?:' + '|'.join(SUFFIXES) + r')\.?$', re.I)

def basic_norm(s):
    if not isinstance(s, str):
        return ''
    s = s.lower()
    s = re.sub(r'[^\w\s]', ' ', s)          # drop punctuation
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def suffix_norm(s):
    s = basic_norm(s)
    # repeatedly strip trailing suffixes
    prev = None
    while prev != s:
        prev = s
        s = SUFFIX_RE.sub('', s).strip()
    return s

# ---------- 5. Rates ----------
n = len(both)
exact = (both['s1_name'] == both['matched_name']).sum()
lower = (both['s1_name'].str.lower() == both['matched_name'].str.lower()).sum()

both['s1_basic'] = both['s1_name'].map(basic_norm)
both['m_basic']  = both['matched_name'].map(basic_norm)
basic = (both['s1_basic'] == both['m_basic']).sum()

both['s1_suf'] = both['s1_name'].map(suffix_norm)
both['m_suf']  = both['matched_name'].map(suffix_norm)
suf = (both['s1_suf'] == both['m_suf']).sum()

print("\n=== Name match rates (among pairs with both names) ===")
print(f"Exact:              {exact:>10,} / {n:,}  ({100*exact/n:.2f}%)")
print(f"Lowercase:          {lower:>10,} / {n:,}  ({100*lower/n:.2f}%)")
print(f"Basic norm:         {basic:>10,} / {n:,}  ({100*basic/n:.2f}%)")
print(f"Suffix-normalized:  {suf:>10,} / {n:,}  ({100*suf/n:.2f}%)")

# ---------- 6. Residual mismatches for inspection ----------
residual = both[both['s1_suf'] != both['m_suf']][
    ['source1_entity_id', 'matched_id', 's1_name', 'matched_name', 's1_suf', 'm_suf']
].sample(min(30, (both['s1_suf'] != both['m_suf']).sum()), random_state=42)

print("\n=== 30 residual mismatches after suffix norm (sample) ===")
pd.set_option('display.max_colwidth', 80)
print(residual.to_string(index=False))

Loading S1 names...
  S1 names: 2,206,821
Loading S2 names...
  S2 names: 5,034,616
Loading S3 names...
  S3 names: 5,285,603

True match pairs after explode: 7,638,365
Pairs with both names present: 7,638,365
  missing S1 name: 0
  missing matched name: 0

=== Name match rates (among pairs with both names) ===
Exact:                 354,118 / 7,638,365  (4.64%)
Lowercase:             821,025 / 7,638,365  (10.75%)
Basic norm:          1,668,793 / 7,638,365  (21.85%)
Suffix-normalized:   2,973,984 / 7,638,365  (38.93%)

=== 30 residual mismatches after suffix norm (sample) ===
source1_entity_id   matched_id                              s1_name                                     matched_name                               s1_suf                                       m_suf
     S1-365897630 S2-942186614             Royal It Private Limited                       रॉयल आईटी प्राइवेट लिमिटेड                             royal it                   र यल आईट प र इव ट ल म ट ड
      S1-65925917  S2

# Address Structure Check

In [ ]:
# Address structure / landmark analysis on true-match pairs
# (assumes DATA_DIR, paths, CHUNK_SIZE already defined)

# ---------- 1. Build address lookups (chunked) ----------
def build_addr_lookup(path):
    lookup = {}
    for chunk in pd.read_csv(path, sep='\t',
                             usecols=['entity_id', 'business_address'],
                             dtype=str, chunksize=CHUNK_SIZE):
        chunk = chunk.dropna(subset=['entity_id'])
        for eid, addr in zip(chunk['entity_id'], chunk['business_address']):
            if eid not in lookup:
                lookup[eid] = addr if pd.notna(addr) else ''
    return lookup

print("Loading addresses...")
s1_addr = build_addr_lookup(S1_PATH)
s2_addr = build_addr_lookup(S2_PATH)
s3_addr = build_addr_lookup(S3_PATH)
print(f"  S1: {len(s1_addr):,}  S2: {len(s2_addr):,}  S3: {len(s3_addr):,}")

# ---------- 2. Explode GT → true pairs (same as name-noise) ----------
gt = pd.read_csv(GT_PATH, sep='\t', dtype=str)
gt['matched_list'] = gt['matched_entity_ids'].fillna('').str.split(',')
pairs = gt.explode('matched_list')
pairs = pairs[pairs['matched_list'].str.len() > 0].copy()
pairs = pairs.rename(columns={'matched_list': 'matched_id'})
print(f"True match pairs: {len(pairs):,}")

# ---------- 3. Attach addresses ----------
def get_addr(mid):
    if mid.startswith('S2-'):
        return s2_addr.get(mid, None)
    if mid.startswith('S3-'):
        return s3_addr.get(mid, None)
    return None

pairs['s1_addr'] = pairs['source1_entity_id'].map(s1_addr)
pairs['matched_addr'] = pairs['matched_id'].map(get_addr)
pairs['source'] = pairs['matched_id'].str[:2]          # 'S2' or 'S3'

both = pairs.dropna(subset=['s1_addr', 'matched_addr']).copy()
print(f"Pairs with both addresses: {len(both):,}")

# ---------- 4. Structural flags ----------
LANDMARK_CUES = re.compile(
    r'\b(near|opp|opposite|behind|next to|beside|landmark|temple|school|hospital|'
    r'church|mosque|mandir|market|bazaar|road|rd|street|st|lane|ln|nagar|colony|'
    r'apartment|apt|floor|fl|building|bldg|tower|complex|plaza|mall|park|garden|'
    r'cross|signal|junction|circle|chowk|gate|main)\b',
    re.I
)
ZIP_LIKE = re.compile(r'\b\d{5}(?:-\d{4})?\b|\b\d{6}\b')   # US ZIP or India PIN
STARTS_DIGIT = re.compile(r'^\s*\d')

def flags(addr):
    if not isinstance(addr, str) or addr.strip() == '':
        return {
            'missing': True,
            'starts_digit': False,
            'has_landmark_cue': False,
            'has_zip_like': False,
            'length': 0
        }
    return {
        'missing': False,
        'starts_digit': bool(STARTS_DIGIT.search(addr)),
        'has_landmark_cue': bool(LANDMARK_CUES.search(addr)),
        'has_zip_like': bool(ZIP_LIKE.search(addr)),
        'length': len(addr)
    }

# Apply to S1 and matched
s1_flags = both['s1_addr'].map(flags).apply(pd.Series)
m_flags  = both['matched_addr'].map(flags).apply(pd.Series)
s1_flags = s1_flags.add_prefix('s1_')
m_flags  = m_flags.add_prefix('m_')
both = pd.concat([both, s1_flags, m_flags], axis=1)

# ---------- 5. Compare S2 vs S3 (and vs S1) ----------
print("\n=== Missing rate ===")
print(both.groupby('source')['m_missing'].mean().rename('matched_missing_rate'))
print("S1 missing rate (on these pairs):", both['s1_missing'].mean())

print("\n=== Starts with digit (street-number style) ===")
print(both.groupby('source')['m_starts_digit'].mean().rename('matched_starts_digit'))
print("S1 starts_digit:", both['s1_starts_digit'].mean())

print("\n=== Has landmark-style cue word ===")
print(both.groupby('source')['m_has_landmark_cue'].mean().rename('matched_landmark_cue'))
print("S1 landmark_cue:", both['s1_has_landmark_cue'].mean())

print("\n=== Has ZIP/PIN-like token ===")
print(both.groupby('source')['m_has_zip_like'].mean().rename('matched_zip_like'))
print("S1 zip_like:", both['s1_has_zip_like'].mean())

print("\n=== Average length ===")
print(both.groupby('source')['m_length'].mean().rename('matched_avg_len'))
print("S1 avg_len:", both['s1_length'].mean())

# ---------- 6. Sample pairs where addresses differ a lot ----------
both['addr_basic_equal'] = (
    both['s1_addr'].str.lower().str.replace(r'[^\w\s]', ' ', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()
    ==
    both['matched_addr'].str.lower().str.replace(r'[^\w\s]', ' ', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()
)

diff = both[~both['addr_basic_equal']].copy()
sample = diff[['source1_entity_id', 'matched_id', 'source',
               's1_addr', 'matched_addr',
               's1_starts_digit', 'm_starts_digit',
               's1_has_landmark_cue', 'm_has_landmark_cue']].sample(
    min(25, len(diff)), random_state=42
)

print("\n=== 25 residual address mismatches (sample) ===")
pd.set_option('display.max_colwidth', 90)
print(sample.to_string(index=False))

Loading addresses...
  S1: 2,206,821  S2: 5,034,616  S3: 5,285,603
True match pairs: 7,638,365
Pairs with both addresses: 7,638,365
